# Scraping con Lyrics.ovh API


## Imports y configuración de rutas

In [ ]:
import sys, os, warnings, importlib.util
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from bson import ObjectId
from datetime import datetime, timezone

# Raiz del proyecto
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR

# Rutas de datos
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
DATA_FIGURES   = PROJECT_ROOT / 'data' / 'figures'
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
DATA_FIGURES.mkdir(parents=True, exist_ok=True)

P1_CSV       = DATA_PROCESSED / 'lyrics_clean.csv'
SCRAPED_CSV  = DATA_PROCESSED / 'lyrics_scraped_api.csv'
COMBINED_CSV = DATA_PROCESSED / 'lyrics_combined.csv'
ENGLISH_CSV  = DATA_PROCESSED / 'lyrics_english.csv'

print(f'Raiz del proyecto : {PROJECT_ROOT}')
print(f'lyrics_clean.csv  : {P1_CSV}  ->  existe={P1_CSV.exists()}')

# Agregar src al path
src_path = str(PROJECT_ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Helper: cargar modulo desde archivo
def _load_module(name, *candidate_paths):
    for path in candidate_paths:
        p = Path(path)
        if p.exists():
            spec = importlib.util.spec_from_file_location(name, str(p))
            mod  = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(mod)
            print(f'{name} cargado desde: {p}')
            return mod
    raise FileNotFoundError(f'No se encontro {name}.py en: {list(candidate_paths)}')

# Cargar scraper_api
scraper_mod = _load_module(
    'scraper_api',
    PROJECT_ROOT / 'src' / 'scraper_api.py',
    PROJECT_ROOT / 'src' / 'data' / 'scraper_api.py',
)
scrape_lyrics_ovh = scraper_mod.scrape_lyrics_ovh
SONGS_BY_GENRE    = scraper_mod.SONGS_BY_GENRE

# Cargar mongo_storage
mongo_mod = _load_module(
    'mongo_storage',
    PROJECT_ROOT / 'src' / 'mongo_storage.py',
    PROJECT_ROOT / 'src' / 'data' / 'mongo_storage.py',
)
insertar_canciones_csv = mongo_mod.insertar_canciones_csv

total_catalogo = sum(len(v) for v in SONGS_BY_GENRE.values())
print(f'\nScraper cargado OK')
print(f'Generos          : {list(SONGS_BY_GENRE.keys())}')
print(f'Total en catalogo: {total_catalogo} canciones')
for g, songs in SONGS_BY_GENRE.items():
    print(f'  {g:<15}: {len(songs)} canciones')

## Cargar corpus del Proyecto 1

In [ ]:
if not P1_CSV.exists():
    raise FileNotFoundError(f'No se encontro: {P1_CSV}\nCopia lyrics_clean.csv a: {DATA_PROCESSED}')

df_p1 = pd.read_csv(P1_CSV)
if 'Source' not in df_p1.columns:
    df_p1['Source'] = 'proyecto1_kaggle'
if 'URL' not in df_p1.columns:
    df_p1['URL'] = None
if 'Language' not in df_p1.columns:
    df_p1['Language'] = 'en'

print(f'Corpus P1 cargado: {len(df_p1):,} canciones')
print(df_p1['Genre'].value_counts().to_string())
df_p1.head(3)

## Probar la API

In [ ]:
import requests

url  = 'https://api.lyrics.ovh/v1/Nirvana/Smells Like Teen Spirit'
resp = requests.get(url, timeout=10)
print(f'Status: {resp.status_code}')

if resp.status_code == 200:
    letra = resp.json().get('lyrics', '')[:300]
    print(f'Primeros 300 caracteres:\n{letra}')
    print('\nAPI funciona correctamente!')
else:
    print(f'Error: {resp.text}')

## Configurar el modo de descarga

In [ ]:
# =====================================================
#  SELECCIONA EL MODO:
#
#  'sample'   ->  ~100 canciones  (5-8 min)    prueba rapida
#  'medio'    ->  ~600 canciones  (30-45 min)  entrega parcial
#  'completo' -> ~1100 canciones  (60-90 min)  cumple los 1000 requeridos
# =====================================================
MODE = 'completo'
# =====================================================

configs = {
    'sample':   dict(max_per_genre=10,  max_total=100),
    'medio':    dict(max_per_genre=60,  max_total=600),
    'completo': dict(max_per_genre=120, max_total=1100),
}

cfg = configs[MODE]
total_cat = sum(len(v) for v in SONGS_BY_GENRE.values())

print(f'Modo seleccionado : {MODE}')
print(f'max_per_genre     : {cfg["max_per_genre"]}')
print(f'max_total         : {cfg["max_total"]}')
print(f'Canciones en catalogo: {total_cat}')
print(f'Exito esperado (~80%): ~{int(min(total_cat, cfg["max_total"]) * 0.8)} canciones')

## Ejecutar la descarga

In [ ]:
FORCE_REDOWNLOAD = True   # False = reusar CSV guardado si ya existe

if SCRAPED_CSV.exists() and not FORCE_REDOWNLOAD:
    df_scraped = pd.read_csv(SCRAPED_CSV)
    print(f'Cargado descarga previa: {len(df_scraped):,} canciones')
else:
    print(f'Iniciando descarga - modo: {MODE}')
    print(f'Objetivo: {cfg["max_total"]} canciones')
    print('=' * 60)

    try:
        df_scraped = scrape_lyrics_ovh(
            songs_by_genre=SONGS_BY_GENRE,
            max_per_genre=cfg['max_per_genre'],
            max_total=cfg['max_total'],
            existing_df=df_p1,
        )
    except KeyboardInterrupt:
        print('\nInterrumpido por el usuario. Guardando lo descargado...')
        df_scraped = pd.DataFrame()

    print('=' * 60)

    if not df_scraped.empty:
        df_scraped.to_csv(SCRAPED_CSV, index=False, encoding='utf-8')
        print(f'\nDescarga completada: {len(df_scraped):,} canciones nuevas')
        print(f'Guardado en: {SCRAPED_CSV}')
        print()
        print('Distribucion por genero:')
        print(df_scraped['Genre'].value_counts().to_string())
    else:
        print('No se obtuvieron canciones. Verifica la conexion a internet.')
        df_scraped = pd.DataFrame()

if not df_scraped.empty:
    display(df_scraped[['Song','Artist','Genre']].head(8))

## Guardar canciones scrapeadas en MongoDB

In [ ]:
if not df_scraped.empty:
    print(f'Insertando {len(df_scraped):,} canciones scrapeadas en MongoDB...')

    df_mongo = df_scraped.copy()

    if '_id' not in df_mongo.columns:
        df_mongo['_id'] = [str(ObjectId()) for _ in range(len(df_mongo))]
    if 'collection_date' not in df_mongo.columns:
        df_mongo['collection_date'] = datetime.now(timezone.utc).strftime('%d-%m-%Y')
    if 'Song year' not in df_mongo.columns:
        df_mongo['Song year'] = 0
    df_mongo['Song year'] = pd.to_numeric(df_mongo['Song year'], errors='coerce').fillna(0).astype(int)

    if 'URL' in df_mongo.columns and 'Url' not in df_mongo.columns:
        df_mongo = df_mongo.rename(columns={'URL': 'Url'})

    resultado = insertar_canciones_csv(df_mongo)
    print(f'  Insertadas  : {resultado["insertados"]:,}')
    print(f'  Ya existian : {resultado["actualizados"]:,}')
    print('Canciones scrapeadas guardadas en MongoDB')
else:
    print('Sin canciones para guardar en MongoDB.')

## Estadisticas del scraping

In [ ]:
if df_scraped.empty:
    print('Sin datos. Saltando graficos.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    gc      = df_scraped['Genre'].value_counts()
    colores = plt.cm.tab10(range(len(gc)))
    bars    = axes[0].bar(gc.index, gc.values, color=colores, edgecolor='white')
    axes[0].set_title('Canciones descargadas por Genero', fontweight='bold')
    axes[0].set_ylabel('Cantidad')
    axes[0].tick_params(axis='x', rotation=35)
    for bar, v in zip(bars, gc.values):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    str(v), ha='center', va='bottom', fontsize=9)

    lens = df_scraped['Lyrics'].str.len().dropna()
    axes[1].hist(lens, bins=30, color='#4C72B0', edgecolor='white')
    axes[1].axvline(lens.mean(), color='red', linestyle='--',
                   label=f'Media: {lens.mean():.0f} chars')
    axes[1].set_title('Longitud de letras', fontweight='bold')
    axes[1].set_xlabel('Caracteres')
    axes[1].set_ylabel('Frecuencia')
    axes[1].legend()

    plt.suptitle(f'Lyrics.ovh API - {len(df_scraped):,} canciones descargadas',
                fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(DATA_FIGURES / 'scraping_stats.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Promedio de caracteres por letra: {lens.mean():.0f}')

## Unir corpus y guardar CSVs finales

In [ ]:
COLS_BASE = ['Song', 'Song year', 'Artist', 'Genre', 'Lyrics', 'Source', 'URL', 'Language']

def normalizar(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # Unificar URL / Url
    if 'Url' in df.columns and 'URL' not in df.columns:
        df = df.rename(columns={'Url': 'URL'})
    for col in COLS_BASE:
        if col not in df.columns:
            df[col] = None
    return df[COLS_BASE].copy()

df_p1_n = normalizar(df_p1)
df_sc_n = normalizar(df_scraped) if not df_scraped.empty else pd.DataFrame(columns=COLS_BASE)

df_combined = pd.concat([df_p1_n, df_sc_n], ignore_index=True)

antes = len(df_combined)
df_combined.drop_duplicates(subset=['Song', 'Artist'], inplace=True)
df_combined.dropna(subset=['Lyrics'], inplace=True)
df_combined = df_combined[df_combined['Lyrics'].str.len() > 50].reset_index(drop=True)

print(f'Duplicados eliminados  : {antes - len(df_combined)}')
print(f'Dataset combinado      : {len(df_combined):,} canciones')
print(f'  Proyecto 1           : {(df_combined["Source"] == "proyecto1_kaggle").sum():,}')
print(f'  Lyrics.ovh API       : {(df_combined["Source"] == "lyrics_ovh_api").sum():,}')

df_english = df_combined[df_combined['Language'].isin(['en', 'unknown'])].reset_index(drop=True)
print(f'  Solo ingles          : {len(df_english):,}')

df_combined.to_csv(COMBINED_CSV, index=False, encoding='utf-8')
df_english.to_csv(ENGLISH_CSV,  index=False, encoding='utf-8')

print(f'\nArchivos guardados:')
print(f'  {COMBINED_CSV.name}  -> {len(df_combined):,} canciones')
print(f'  {ENGLISH_CSV.name}   -> {len(df_english):,} canciones')

nuevas = (df_combined['Source'] == 'lyrics_ovh_api').sum()
estado = 'REQUISITO CUMPLIDO' if nuevas >= 1000 else f'FALTAN {1000 - nuevas} CANCIONES'
print(f'\n{estado}: {nuevas:,} canciones nuevas (minimo requerido: 1,000)')
print('\n-> Siguiente paso: 03_bert_analysis.ipynb')

## Guardar corpus combinado en MongoDB

In [ ]:
print(f'Sincronizando corpus combinado en MongoDB ({len(df_combined):,} canciones)...')

df_mongo_all = df_combined.copy()

if '_id' not in df_mongo_all.columns:
    df_mongo_all['_id'] = [str(ObjectId()) for _ in range(len(df_mongo_all))]
if 'collection_date' not in df_mongo_all.columns:
    df_mongo_all['collection_date'] = datetime.now(timezone.utc).strftime('%d-%m-%Y')
if 'Song year' not in df_mongo_all.columns:
    df_mongo_all['Song year'] = 0
df_mongo_all['Song year'] = pd.to_numeric(df_mongo_all['Song year'], errors='coerce').fillna(0).astype(int)

if 'URL' in df_mongo_all.columns and 'Url' not in df_mongo_all.columns:
    df_mongo_all = df_mongo_all.rename(columns={'URL': 'Url'})

resultado_all = insertar_canciones_csv(df_mongo_all)
print(f'  Insertadas  : {resultado_all["insertados"]:,}')
print(f'  Ya existian : {resultado_all["actualizados"]:,}')
print('Corpus combinado sincronizado en MongoDB')
print('  Base de datos: musica  |  Coleccion: canciones')

## Grafico resumen final

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

gc_all  = df_combined['Genre'].value_counts()
colores = plt.cm.tab10(range(len(gc_all)))
bars    = axes[0].bar(gc_all.index, gc_all.values, color=colores, edgecolor='white')
axes[0].set_title('Dataset Combinado por Genero', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Canciones')
axes[0].tick_params(axis='x', rotation=35)
for bar, v in zip(bars, gc_all.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(v), ha='center', va='bottom', fontsize=9)

fuentes   = df_combined['Source'].value_counts()
etiquetas = {'proyecto1_kaggle': 'Proyecto 1\n(Kaggle)', 'lyrics_ovh_api': 'Lyrics.ovh\nAPI'}
axes[1].pie(
    fuentes.values,
    labels=[f"{etiquetas.get(k, k)}\n({v:,})" for k, v in fuentes.items()],
    autopct='%1.1f%%',
    colors=['#4C72B0', '#DD8452'],
    startangle=90,
)
axes[1].set_title('Fuente de los datos', fontweight='bold', fontsize=12)

plt.suptitle(f'Corpus Final: {len(df_combined):,} canciones', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(DATA_FIGURES / 'corpus_final_stats.png', dpi=120, bbox_inches='tight')
plt.show()